# 4. Entraînement réel d'un TCN causal sur CPU

**Question.** Le passé local permet-il d'estimer qu'une confirmation physique est disponible maintenant ?

Ce notebook charge les résultats effectivement obtenus. Son exécution normale ne lance aucun entraînement. Le TCN ne prédit ni une étiquette future fournie en entrée, ni une action de récupération. Il utilise des convolutions causales dilatées et un masque de préfixe ; les 7 dilatations donnent un champ réceptif de 255 mesures, suffisant pour couvrir les 176 mesures possibles.


In [ ]:
from pathlib import Path
import os
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import HTML, display
from cav_recovery.laboratory.workbench import Laboratory
from cav_recovery.laboratory.storage import json_file

lab = Laboratory.from_environment()
display(HTML(lab.table_html([{'résultats': str(lab.root), 'M02': str(lab.dataset)}])))


## Protocole gelé avant l'accès au test

Les poids sont ajustés sur train ; les statistiques de normalisation sont ajustées sur train uniquement. Deux tailles TCN sont comparées sur la BCE de validation, avec une baseline instantanée. Le seuil statistique est calibré sur validation. Le test OOD M02 n'intervient ni dans la sélection des poids, ni dans celle de l'architecture ou du seuil.


In [ ]:
protocole = json_file(lab.campaign / 'protocol.json')
selection = json_file(lab.campaign / 'selected-model.json')
display(HTML(lab.table_html(protocole['trials'])))
display(HTML(lab.table_html([
    {'paramètre': cle, 'valeur': protocole[cle]}
    for cle in ['history_samples','sample_s','seed','threads','batch_size','epochs',
                'train_stride','validation_selection_stride','learning_rate',
                'dilations','kernel_size','checkpoint_steps','patience']
])))
display(HTML(lab.table_html([selection])))
fig = lab.plot_losses()
plt.show()


## Architecture, normalisation et modèle réellement sauvegardé

Les mêmes fonctions construisent le modèle dans la CLI et dans le notebook. Les canaux bruts n'incluent ni identifiants, ni coordonnées absolues, ni statut D01/C-RDG. La baseline instantanée reçoit les mêmes variables au dernier instant ; elle permet de mesurer l'intérêt de l'historique, sans changer la tâche.


In [ ]:
from cav_recovery.laboratory.model import create_model
modele_dir = lab.campaign / 'm02' / selection['selected']
configuration = json_file(modele_dir / 'config.json')
resultat = json_file(modele_dir / 'complete.json')
modele = create_model(configuration, resultat['compatibility']['features'])
from cav_recovery.laboratory.checkpoints import CheckpointStore
etat, preuve = CheckpointStore(modele_dir / 'checkpoints', resultat['compatibility']).load(best=True)
modele.load_state_dict(etat['model'])
modele.eval()
print(modele)
display(HTML(lab.table_html([{'paramètres': resultat['parameters'], 'époques terminées': resultat['completed_epochs'],
                            'fenêtres train utilisées': resultat['train_windows'],
                            'identité données': resultat['compatibility']['dataset']}])) )
print('Empreinte du checkpoint chargé :', preuve['receipt']['sha256'])
print('Durée murale enregistrée (s) :', resultat['wall_s'])
anomalies = lab.root / 'timing-anomalies.json'
if anomalies.exists():
    display(HTML(lab.table_html([json_file(anomalies)])))
from cav_recovery.laboratory.training import WindowData, predict
validation = WindowData(lab.cache, 'validation')
indices_exemples = validation.indices()[:8]
probabilites, _ = predict(modele, validation, indices_exemples)
display(HTML(lab.table_html([
    {'instant (s)': float(validation.windows['time_s'][i]),
     'probabilité calculée': float(p), 'cible indépendante': int(validation.windows['y'][i])}
    for i,p in zip(indices_exemples, probabilites)
])))


## Coupure et reprise réellement exercées

Les checkpoints contiennent les poids, l'optimiseur, les RNG, l'époque, le curseur du mélange, les métriques et l'identité des données. Chaque génération est validée avant publication atomique du pointeur ; une précédente génération valide reste disponible. Le premier entraînement réel a subi une sortie brutale du processus après trois lots, puis a repris au curseur enregistré.

La garantie est une reprise au dernier lot sauvegardé. Les lots non sauvegardés peuvent être recalculés. Cette infrastructure pourra conserver des états supplémentaires Graph-MARL ; elle ne promet pas une restauration d'un épisode physique en cours.


In [ ]:
preuve_reprise = json_file(lab.campaign / 'power-cut-proof.json')
display(HTML(lab.table_html([preuve_reprise])))
assert preuve_reprise['completed'] and preuve_reprise['actual_process_terminated']
assert preuve_reprise['resume_records'] and preuve_reprise['stale_locks_preserved']


## Résultats sur le test OOD réservé

Les fenêtres corrélées ne sont pas autant d'expériences indépendantes. Consulter aussi les métriques par épisode, les inconnus masqués et les événements manqués. Une estimation correcte de confirmation maintenant ne démontre pas une anticipation future.


In [ ]:
display(HTML(lab.table_html(lab.model_results('m02'))))
fig = lab.plot_confusion('m02')
plt.show()


## Mode pédagogique facultatif

Le code suivant appelle le même entraîneur, dans un nouveau dossier, avec une seule époque et un échantillonnage plus léger. Laisser ce mode désactivé pour une présentation de consultation. Il ne remplace pas le modèle sélectionné et ne change aucun résultat historique.


In [ ]:
LANCER_DEMONSTRATION = False
if LANCER_DEMONSTRATION:
    from datetime import datetime
    from cav_recovery.laboratory.training import train
    configuration_legere = dict(configuration, epochs=1, train_stride=100,
                                validation_selection_stride=100, checkpoint_steps=10)
    sortie_distincte = lab.root / ('pedagogie-' + datetime.now().strftime('%Y%m%d-%H%M%S'))
    resultat_leger = train(lab.cache, sortie_distincte, configuration_legere)
    display(HTML(lab.table_html([{'dossier distinct': str(sortie_distincte),
                                'époques': resultat_leger['completed_epochs']}])) )
